# Deep Learning: Regression & Classification
1. Stock Closing Price Prediction (BiLSTM + Multi-Head Attention)
2. Emotion Classification (fine-tuned ERNIE 2.0)

# 1. Stock Closing Price Prediction

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
)
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
DATA_PATH = "../data/X.csv"
RESULT_DIR = "../result"
MODEL_PATH = f"{RESULT_DIR}/model/stock_model.keras"
WINDOW = 20  # days of history per sample
SPLIT = (0.8, 0.9)  # train / val / test boundaries (chronological)
EPOCHS, BATCH, SEED = 100, 32, 42

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
keras.utils.set_random_seed(SEED)

In [ ]:
# Load, clean, sort
df = (
    pd.read_csv(DATA_PATH, usecols=["Date", "Close"], parse_dates=["Date"])
    .dropna()
    .sort_values("Date")
    .reset_index(drop=True)
)
print(df.describe())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df["Close"], bins=50)
ax[0].set_title("Close Distribution")
ax[1].plot(df["Date"], df["Close"])
ax[1].set_title("Closing Price Over Time")

fig.savefig(f"{RESULT_DIR}/stock_eda.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scale (fit on train only to avoid leakage) and build sliding windows
close = df[["Close"]].values
i_tr, i_va = (int(len(close) * s) for s in SPLIT)
scaler = MinMaxScaler().fit(close[:i_tr])
scaled = scaler.transform(close).astype("float32")


def make_windows(start, end):
    idx = range(max(start, WINDOW), end)
    return np.stack([scaled[i - WINDOW : i] for i in idx]), np.stack(
        [scaled[i] for i in idx]
    )


(x_tr, y_tr), (x_va, y_va), (x_te, y_te) = (
    make_windows(0, i_tr),
    make_windows(i_tr, i_va),
    make_windows(i_va, len(scaled)),
)
print(x_tr.shape, x_va.shape, x_te.shape)

In [ ]:
# Model predicts the change from the last observed price (residual), which generalizes beyond the train price range
inp = keras.Input((WINDOW, 1))
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inp)
x = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
x = layers.Dense(1)(layers.GlobalAveragePooling1D()(x))
model = keras.Model(inp, layers.Add()([x, inp[:, -1, :]]))
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])

callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True),
]
hist = model.fit(
    x_tr,
    y_tr,
    validation_data=(x_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
# Evaluate in original price scale (naive "tomorrow = today" baseline for reference)
inv = scaler.inverse_transform
pred, true, naive = inv(model.predict(x_te, verbose=0)), inv(y_te), inv(x_te[:, -1])
for name, p in [("Model", pred), ("Naive", naive)]:
    print(
        f"{name}: MAE={mean_absolute_error(true, p):.3f} RMSE={root_mean_squared_error(true, p):.3f} MAPE={mean_absolute_percentage_error(true, p):.2%}"
    )

plt.figure(figsize=(9, 5))
plt.plot(true, label="Actual")
plt.plot(pred, label="Predicted")
plt.title("Stock Closing Price Prediction (Test Set)")
plt.xlabel("Time Step")
plt.ylabel("Close")
plt.legend()
plt.grid()

plt.savefig(f"{RESULT_DIR}/stock_prediction.png", bbox_inches="tight")
plt.show()

In [ ]:
# Inference: predict next day's close from the latest WINDOW days
best = keras.models.load_model(MODEL_PATH)
last = scaler.transform(close[-WINDOW:]).astype("float32")[None]
print(
    f"Last close: {close[-1, 0]:.3f} | Predicted next close: {inv(best.predict(last, verbose=0))[0, 0]:.3f}"
)

# 2. Emotion Classification with ERNIE

In [ ]:
import os
import re

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

In [ ]:
DATA_PATH = "../data/Emotion.csv"
RESULT_DIR = "../result"
MODEL_DIR = f"{RESULT_DIR}/model/emotion_ernie"
HEAD_PATH = f"{MODEL_DIR}/head.joblib"  # CPU mode only
MODEL_NAME = "nghuyong/ernie-2.0-base-en"
MAX_LEN, BATCH, EPOCHS, LR, SEED = 64, 32, 3, 2e-5, 42
GPU = (
    torch.cuda.is_available()
)  # GPU: full fine-tune + AMP | CPU: frozen embeddings + LogisticRegression

DEVICE = torch.device("cuda" if GPU else "cpu")
os.makedirs(MODEL_DIR, exist_ok=True)
torch.manual_seed(SEED)

In [ ]:
# Load, clean, dedupe
clean = lambda t: re.sub(r"[^a-z0-9\s]", "", str(t).lower())
df = pd.read_csv(DATA_PATH).dropna().drop_duplicates()
df["text"] = df["text"].map(clean)
classes = sorted(df["label"].unique())
label2id = {c: i for i, c in enumerate(classes)}
df["y"] = df["label"].map(label2id)

df["label"].value_counts().plot.bar(figsize=(6, 3), title="Label Distribution")
plt.savefig(f"{RESULT_DIR}/emotion_labels.png", bbox_inches="tight")
plt.show()

In [ ]:
# Stratified 70/15/15 split; tokenize with dynamic padding (much faster than padding to the global max)
tr, tmp = train_test_split(df, test_size=0.3, stratify=df["y"], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.5, stratify=tmp["y"], random_state=SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collate = DataCollatorWithPadding(tokenizer)


def make_loader(d, shuffle=False):
    enc = tokenizer(d["text"].tolist(), truncation=True, max_length=MAX_LEN)
    ds = [
        {**{k: v[i] for k, v in enc.items()}, "labels": y} for i, y in enumerate(d["y"])
    ]
    
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, collate_fn=collate)


train_dl, val_dl, test_dl = make_loader(tr, True), make_loader(va), make_loader(te)

In [ ]:
# GPU: fine-tune ERNIE with mixed precision, warmup + linear decay, keep best epoch by val macro-F1
# CPU: encode texts once with frozen ERNIE (mean pooling), then fit LogisticRegression
def pool(out, mask):
    m = mask.unsqueeze(-1).to(out.last_hidden_state.dtype)
    return (out.last_hidden_state * m).sum(1) / m.sum(1)


@torch.inference_mode()
def predict(dl):
    model.eval()
    preds, trues = [], []
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        y = b.pop("labels")
        with torch.autocast(DEVICE.type, enabled=GPU):
            if GPU:
                preds.append(model(**b).logits.argmax(-1).cpu())
            else:
                preds.append(pool(model(**b), b["attention_mask"]).float())

        trues.append(y.cpu())
        
    return torch.cat(trues).numpy(), torch.cat(preds).numpy()


if GPU:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(classes),
        id2label=dict(enumerate(classes)),
        label2id=label2id,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(train_dl) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler()

    best_f1 = 0
    for epoch in range(EPOCHS):
        model.train()
        for b in train_dl:
            with torch.autocast("cuda"):
                loss = model(**{k: v.to(DEVICE) for k, v in b.items()}).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        f1 = f1_score(*predict(val_dl), average="macro")
        print(f"Epoch {epoch + 1}: loss={loss.item():.4f} val_macro_f1={f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
else:
    torch.set_num_threads(os.cpu_count())
    model = AutoModel.from_pretrained(MODEL_NAME)

    (y_tr, X_tr), (y_va, X_va), (y_te, X_te) = map(predict, (train_dl, val_dl, test_dl))
    head = LogisticRegression(max_iter=2000, C=2.0, class_weight="balanced").fit(
        X_tr, y_tr
    )

    print(f"val_macro_f1={f1_score(y_va, head.predict(X_va), average='macro'):.4f}")

    model.save_pretrained(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)
    joblib.dump(head, HEAD_PATH)

In [ ]:
# Evaluate best model on test set
if GPU:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
    y_true, y_pred = predict(test_dl)
else:
    y_true, y_pred = y_te, head.predict(X_te)

print(classification_report(y_true, y_pred, target_names=classes, digits=3))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classes, xticks_rotation=45, cmap="Blues"
)

plt.savefig(f"{RESULT_DIR}/emotion_confusion_matrix.png", bbox_inches="tight")
plt.show()

In [ ]:
# Inference (loads saved model from disk)
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
if GPU:
    model = (
        AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()
    )
else:
    model, head = AutoModel.from_pretrained(MODEL_DIR).eval(), joblib.load(HEAD_PATH)


@torch.inference_mode()
def classify(texts):
    enc = tok(
        [clean(t) for t in texts],
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)
    if GPU:
        probs = model(**enc).logits.softmax(-1).cpu().numpy()
    else:
        probs = head.predict_proba(pool(model(**enc), enc["attention_mask"]).numpy())
    return [(classes[p.argmax()], p.max()) for p in probs]


samples = [
    "i feel so happy and grateful today",
    "i am terrified of what might happen next",
]

for s, (label, score) in zip(samples, classify(samples)):
    print(f"{label:>9} ({score:.2f}) | {s}")